# TB-Trust — 05: Fiducial coverage audit (physics track, run this first)

**The load-bearing assumption of the entire physics track is tested here, and nowhere else.**

Every chest radiograph carries three physical calibration targets, and each buys one of the
three unknowns in a phone photograph of that film:

| object on the film | known optical density | what it measures |
|---|---|---|
| lead L/R side marker | `D_min` (base+fog) | the bright densitometry anchor |
| direct-exposure region | `D_max` | the veiling-glare **beam stop** |
| collimation border | a hard `D_min`→`D_max` step | the capture **PSF**, via ISO 12233 |

A note on signs, because it is the thing everyone gets backwards. Lead blocks X-rays, so the
film beneath it is barely exposed, so it is *transparent* — the **brightest** thing on the
lightbox. The direct-exposure region took the full beam, developed to maximum density, and is
the **darkest**. So the optical beam stop is the direct-exposure region, not the marker. See
`src/tbtrust/physics/density.py`.

None of that helps if the archive threw the fiducials away. Montgomery, Shenzhen, NIAID and
RSNA were each assembled by someone free to crop to the lung fields, and a tight crop removes
all three at once. **This notebook measures how often they survived.** It is cheap — no model,
no GPU, milliseconds per image — and its answer determines what the rest of the track can claim.

In [ ]:
# --- configuration ---------------------------------------------------------
# Every path comes from the environment first, so this notebook runs unmodified
# on Kaggle, locally, or under scripts/test_notebooks.py in CI.
import os

REPO = os.environ.get("TBTRUST_REPO", "/kaggle/working/tb-trust")
DATA = os.environ.get("TBTRUST_DATA", "/kaggle/input/tuberculosis-tb-chest-xray-dataset")
WORK = os.environ.get("TBTRUST_WORK", "/kaggle/working")
REPO_URL = os.environ.get("TBTRUST_REPO_URL", "https://github.com/AIscend-Research/tb-trust.git")

MANIFEST = f"{WORK}/manifest.csv"
OUT = f"{WORK}/outputs"
os.makedirs(OUT, exist_ok=True)

# Working resolution for the physics. This is the single most consequential knob
# in the whole track: the density floor depends on how many pixels a finding
# spans, so a 2 mm miliary nodule is under two pixels at 320 px and the
# certificate correctly -- but uselessly -- calls every image insufficient.
# A phone photographing a 35 cm film at 3000 px gets about 8 px/mm. 1024 is the
# smallest size at which the severity sweep separates properly; drop it only to
# make a CI run cheap.
PHYSICS_SIZE = int(os.environ.get("TBTRUST_PHYSICS_SIZE", "1024"))
N_IMAGES = int(os.environ.get("TBTRUST_PHYSICS_N", "24"))

print("REPO:", REPO, "\nDATA:", DATA, "\nWORK:", WORK)
print("physics size:", PHYSICS_SIZE, " images:", N_IMAGES)

In [ ]:
# Enter the repo and make it importable. The install is skipped when the package
# already resolves, so re-running is cheap.
import importlib.util
import os
import subprocess
import sys

os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
if importlib.util.find_spec("tbtrust") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 110
print("tbtrust ready from", REPO)

In [ ]:
# Build the manifest if an earlier notebook has not already done it.
import subprocess
import sys
from pathlib import Path

if not Path(MANIFEST).exists():
    r = subprocess.run([sys.executable, "scripts/build_manifest.py", "--raw", DATA, "--out", MANIFEST],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
    assert r.returncode == 0, "build_manifest failed"

manifest = pd.read_csv(MANIFEST)
print(len(manifest), "images;", manifest["clinic"].value_counts().to_dict())

## 1. Run the detector over the corpus

`coverage` grades what each image unlocks:

- **full** — marker + beam stop + a usable slanted edge. Everything is measurable.
- **partial** — a beam stop but no marker or no usable edge. Glare is still measured directly
  (it is the dominant term), but the density scale leans on the sRGB gamma prior.
- **none** — not even a dark surround. The certificate must abstain
these images are outside
  the method's reach and must be reported as such rather than quietly dropped.

In [ ]:
import subprocess
import sys

AUDIT = f"{OUT}/fiducial_audit"
r = subprocess.run(
    [sys.executable, "scripts/audit_fiducials.py", "--manifest", MANIFEST, "--out", AUDIT],
    capture_output=True, text=True,
)
print(r.stdout[-3000:] or r.stderr[-3000:])
assert r.returncode == 0, r.stderr[-2000:]

audit = pd.read_csv(f"{AUDIT}.csv")
audit.head()

## 2. The coverage table

This table goes in the paper. It is not a diagnostic — it is a *result*, and it bounds every
claim downstream. If a clinic shows a low certifiable rate, the honest move is to say so and
report the physics results on the certifiable subset with its size stated, not to evaluate on
whichever images happened to work.

In [ ]:
def coverage_table(df):
    g = df.groupby("clinic")
    out = pd.DataFrame({
        "n": g.size(),
        "full": g["coverage"].apply(lambda s: (s == "full").mean()),
        "partial": g["coverage"].apply(lambda s: (s == "partial").mean()),
        "none": g["coverage"].apply(lambda s: (s == "none").mean()),
        "marker": g["has_marker"].mean(),
        "beam stop": g["has_beamstop"].mean(),
        "MTF edge": g["n_mtf_edges"].apply(lambda s: (s > 0).mean()),
    })
    out.loc["ALL"] = [
        len(df),
        (df["coverage"] == "full").mean(), (df["coverage"] == "partial").mean(),
        (df["coverage"] == "none").mean(), df["has_marker"].mean(),
        df["has_beamstop"].mean(), (df["n_mtf_edges"] > 0).mean(),
    ]
    return out.round(3)


cov = coverage_table(audit)
display(cov)

certifiable = float((audit["coverage"] != "none").mean())
print(f"\ncertifiable (a physics bound is available): {certifiable:.1%}")
print("beam stop source:", audit["beamstop_source"].value_counts().to_dict())

## 3. Where the beam stop comes from

Two sources, and the distinction matters for how well the glare *field* is pinned down.

A **collimated rim** is an annulus, so it samples the veil all the way round the frame and
constrains its spatial shape, which is what makes "the glare is upper-left, move the phone"
possible. A **dark surround** — the black area around the patient silhouette, which survives
even an aggressive crop — gives the veil's level but much weaker leverage on its shape.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

src = audit.groupby(["clinic", "beamstop_source"]).size().unstack(fill_value=0)
src.plot(kind="bar", stacked=True, ax=axes[0], rot=20)
axes[0].set_title("Beam-stop source")
axes[0].set_ylabel("images")
axes[0].legend(fontsize=7)

for c, g in audit.groupby("clinic"):
    axes[1].hist(g["marker_confidence"], bins=25, histtype="step", label=f"{c} (n={len(g)})")
axes[1].axvline(0.6, color="k", ls="--", lw=1)
axes[1].set_xlabel("lead-marker confidence")
axes[1].set_title("Marker detection (0.6 = accepted)")
axes[1].legend(fontsize=7)

axes[2].hist(audit["n_mtf_edges"], bins=np.arange(-0.5, 5.5), rwidth=0.8)
axes[2].set_xlabel("usable slanted edges")
axes[2].set_title("Edges available for the MTF")

fig.tight_layout()
plt.show()

## 4. Look at what the detector actually found

Numbers hide detection failures that are obvious by eye. This contact sheet overlays the
collimation quad (cyan), the beam stop (red) and the lead marker (yellow) on a sample of
images. Check that the red region really is the black rim outside the patient and that the
cyan box really is the collimation border — if the detector is latching onto the image
frame instead, every downstream number is meaningless and the coverage table above looks fine.

In [ ]:
from PIL import Image

from tbtrust.physics import fiducials as FID

sample = audit.sample(min(6, len(audit)), random_state=0).reset_index(drop=True)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, row in zip(axes.ravel(), sample.itertuples(), strict=False):
    img = np.asarray(Image.open(row.path).convert("L").resize((512, 512), Image.BILINEAR))
    f = FID.detect(img)
    ax.imshow(img, cmap="gray")
    if f.beamstop_mask is not None:
        ax.imshow(np.ma.masked_where(~f.beamstop_mask, f.beamstop_mask), cmap="autumn", alpha=0.45)
    if f.marker_mask is not None and f.marker_mask.any():
        ax.imshow(np.ma.masked_where(~f.marker_mask, f.marker_mask), cmap="Wistia", alpha=0.9)
    if f.field_quad is not None:
        q = np.vstack([f.field_quad, f.field_quad[:1]])
        ax.plot(q[:, 0], q[:, 1], "c-", lw=1.5)
    ax.set_title(f"{row.clinic} · {f.coverage.value}\nmarker {f.marker_confidence:.2f}, "
                 f"{len(f.mtf_edges)} MTF edges", fontsize=8)
    ax.axis("off")
for ax in axes.ravel()[len(sample):]:
    ax.axis("off")
fig.tight_layout()
plt.show()

## 5. Verdict, and what it licenses

Read the number below and pick the corresponding path. Both are legitimate
only pretending
matters.

In [ ]:
n_cert = int((audit["coverage"] != "none").sum())
print(f"{n_cert}/{len(audit)} images ({certifiable:.1%}) support a physics-derived bound.\n")

if certifiable >= 0.5:
    print("REAL-PHOTO PATH is available.")
    print("  Notebooks 06-08 can run with --real on this corpus, and the certificate is a")
    print("  statement about the archived images themselves.")
else:
    print("SIMULATED RE-PHOTOGRAPHY PATH.")
    print("  Most images were cropped past their fiducials, so a bound cannot be stated for")
    print("  them directly. This is a finding to report with its number, not a bug.")
    print("  Notebooks 06-08 then treat each archive image as a clean film, paint the")
    print("  fiducials back on and re-photograph it through the forward model in")
    print("  physics/film.py. That is a controlled experiment -- capture quality becomes a")
    print("  knob and ground truth exists -- and it is what the in-silico limitation in")
    print("  docs/LIMITATIONS.md already covers.")

audit.to_csv(f"{OUT}/fiducial_audit.csv", index=False)
cov.to_csv(f"{OUT}/fiducial_coverage_table.csv")
print(f"\nwrote {OUT}/fiducial_coverage_table.csv")